# Train the pipe / rebar detector

Press **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom (about 30-60 minutes).
At the end you download one zip with two files; put them in `app/public/models/` of the photo-counter repo (or send them to Claude) and the app starts counting Pipes and Rebar automatically.

You need a free Roboflow API key: roboflow.com → sign in → Settings → API Keys. It is only typed into this notebook, never saved.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run."
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone --depth 1 https://github.com/Phil-Green-89/photo-counter.git
%cd photo-counter
!pip install -q ultralytics onnx onnxslim onnxruntime roboflow numpy pillow

In [ ]:
import os, getpass
os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")

Downloads the public pipe/rebar datasets listed in `training/datasets.json`, merges them into one class, and adds dark/noisy copies of every training image. Datasets that fail to download are skipped with a message; you want at least a few thousand images in total.

In [ ]:
!python training/prepare_data.py data/ --roboflow --synthetic 300

Train. Early stopping is on, so it may finish before 80 epochs.

In [ ]:
!python training/train_pipes.py data/ --epochs 80 --version pipes-v1 --batch 32

Accuracy of the exact file the app will download, at three light levels (1.00 = normal, 0.50 = half the light, 0.25 = a quarter).
Look at the **exact** and **±1** columns: for counting, 90%+ within ±1 at 1.00 and 0.50 is a good target.

In [ ]:
!python training/eval_counts.py data/ app/public/models

In [ ]:
!cd app/public/models && zip -j /content/pipes-model.zip pipes.onnx manifest.json ATTRIBUTION.md
from google.colab import files
files.download("/content/pipes-model.zip")